<a href="https://colab.research.google.com/github/parisazeynaly/Advanced--Statistics-Emprical/blob/main/RL_Exploration_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**An** **Empirical Comparison of Classical and Bayesian Exploration Strategies in RL**


**CORE Q-LEARNING LOOP ON FROZENLAKE**

In [10]:


import numpy as np
import gymnasium as gym
from scipy import stats
from itertools import product
import json
import os
import warnings
warnings.filterwarnings('ignore')

# ── Try to import plotting (optional but recommended) ─────────────────────────
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    PLOT = True
except ImportError:
    PLOT = False
    print("matplotlib not found — skipping plots. Install with: pip install matplotlib")

os.makedirs("results", exist_ok=True)

In [11]:
import numpy as np
import gymnasium as gym
from scipy import stats
from itertools import product
import json
import os
import warnings
warnings.filterwarnings('ignore')

# ── Try to import plotting (optional but recommended) ─────────────────────────
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    PLOT = True
except ImportError:
    PLOT = False
    print("matplotlib not found — skipping plots. Install with: pip install matplotlib")

os.makedirs("results", exist_ok=True)

# HYPERPARAMETERS
ALPHA       = 0.1      # learning rate — how much each update shifts Q
GAMMA       = 0.99     # discount factor — how much future reward matters
EPISODES    = 5000     # total training episodes for single run
EPS_START   = 1.0      # initial exploration rate (100% random at start)
EPS_END     = 0.01     # final exploration rate (1% random at the end)
MAP_NAME    = "4x4"    # FrozenLake grid size for single run: "4x4" or "8x8"
IS_SLIPPERY = False    # False = deterministic, True = stochastic (slips) for single run
SEED        = 42       # for reproducibility
N_RUNS      = 30       # Number of independent runs for each condition in main experiment

CONFIG = {
    "4x4": {"episodes": 5_000,  "map": "4x4"},
    "8x8": {"episodes": 15_000, "map": "8x8"},
}

TAU_START     = 1.0
TAU_END       = 0.01
STABILITY_WIN = 0.20    # last 20% of episodes for stability measure
CONV_THRESH   = 0.70    # reward threshold for convergence
CONV_WINDOW   = 100     # consecutive episodes above threshold
CI_BOOTS      = 2000    # bootstrap samples for confidence intervals
ALPHA_STAT    = 0.05    # significance level

COLORS = {
    "epsilon_greedy": "#E74C3C",
    "softmax":        "#3498DB",
    "thompson":       "#2ECC71",
}
LABELS = {
    "epsilon_greedy": "Epsilon-Greedy",
    "softmax":        "Softmax",
    "thompson":       "Thompson Sampling",
}

EPSILON DECAY

In [12]:
def get_epsilon(episode, total_episodes):
    """
    Linearly decay epsilon from EPS_START to EPS_END over training.

    Why decay? Early on, the agent knows nothing, so it should explore
    a lot (high epsilon). As training progresses, the Q-table becomes
    more reliable, so the agent should exploit what it has learned
    (low epsilon).
    """
    fraction = episode / total_episodes
    epsilon = EPS_START - fraction * (EPS_START - EPS_END)
    return max(EPS_END, epsilon)

ACTION SELECTION (epsilon-greedy — the simplest strategy)

In [13]:
def choose_action(Q, state, epsilon, n_actions, rng):
    """
    With probability epsilon: pick a uniformly random action (explore).
    With probability 1-epsilon: pick the action with the highest known
    Q-value for this state (exploit).
    """
    if rng.random() < epsilon:
        return rng.integers(n_actions)          # explore
    else:
        return int(np.argmax(Q[state]))         # exploit

THE Q-LEARNING UPDATE (the Bellman equation )

In [14]:
def update_q(Q, state, action, reward, next_state, terminated):
    """
    Q(s,a) <- Q(s,a) + alpha * [ r + gamma * max_a' Q(s',a') - Q(s,a) ]

    Read it left to right:
      - "r + gamma * max_a' Q(s',a')"  is the TD TARGET — our new best
        guess of how good (s,a) really is, using the reward we just got
        plus the best value we can get from the next state.
      - "... - Q(s,a)"  is how far off our OLD estimate was. This
        difference is called the TD ERROR.
      - We nudge Q(s,a) toward the target by a small step (alpha).

    If next_state is terminal (hole or goal), there is no future —
    so the "max_a' Q(s',a')" term is zero.
    """
    if terminated:
        td_target = reward
    else:
        td_target = reward + GAMMA * np.max(Q[next_state])

    td_error = td_target - Q[state, action]
    Q[state, action] += ALPHA * td_error

    return td_error   # returned only so we can print/inspect it while learning

In [15]:
def run_episode(env, Q, epsilon, rng):
    """
    Plays one full episode: from reset() until the agent reaches the
    goal, falls in a hole, or the episode times out.

    Returns the total reward collected (0 or 1 for FrozenLake, since
    reward is sparse — only the goal gives +1).
    """
    state, _ = env.reset(seed=int(rng.integers(1_000_000)))
    n_actions = env.action_space.n
    total_reward = 0.0

    while True:
        action = choose_action(Q, state, epsilon, n_actions, rng)
        next_state, reward, terminated, truncated, _ = env.step(action)

        update_q(Q, state, action, reward, next_state, terminated)

        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return total_reward

In [16]:
def train():
    rng = np.random.default_rng(SEED)
    env = gym.make("FrozenLake-v1", map_name=MAP_NAME, is_slippery=IS_SLIPPERY)

    n_states  = env.observation_space.n
    n_actions = env.action_space.n
    Q = np.zeros((n_states, n_actions))

    rewards_per_episode = []

    for ep in range(EPISODES):
        epsilon = get_epsilon(ep, EPISODES)
        reward = run_episode(env, Q, epsilon, rng)
        rewards_per_episode.append(reward)

        # Print progress every 500 episodes so you can watch it learn
        if (ep + 1) % 500 == 0:
            recent_success_rate = np.mean(rewards_per_episode[-500:])
            print(f"Episode {ep+1:5d} | epsilon={epsilon:.3f} | "
                  f"success rate (last 500 eps) = {recent_success_rate:.2%}")

    env.close()
    return Q, rewards_per_episode

SANITY CHECKS

In [17]:
def evaluate(Q, n_eval_episodes=100):
    """
    Run the trained agent with epsilon=0 (pure exploitation, no randomness)
    to see its true success rate. This is what "performance" means.
    """
    env = gym.make("FrozenLake-v1", map_name=MAP_NAME, is_slippery=IS_SLIPPERY)
    rng = np.random.default_rng(123)  # different seed than training
    successes = 0

    for _ in range(n_eval_episodes):
        state, _ = env.reset(seed=int(rng.integers(1_000_000)))
        for _ in range(200):  # safety cap on steps per episode
            action = int(np.argmax(Q[state]))   # always greedy, no exploration
            state, reward, terminated, truncated, _ = env.step(action)
            if terminated or truncated:
                successes += reward
                break

    env.close()
    return successes / n_eval_episodes


def print_q_table(Q):
    """Pretty-print the Q-table so you can inspect it visually."""
    action_names = ["LEFT", "DOWN", "RIGHT", "UP"]
    print("\nFinal Q-table:")
    print("State | " + " | ".join(f"{a:>7s}" for a in action_names))
    for s in range(Q.shape[0]):
        row = " | ".join(f"{Q[s, a]:7.3f}" for a in range(Q.shape[1]))
        print(f"{s:5d} | {row}")


In [18]:
if __name__ == "__main__":
    print(f"Training Q-learning on FrozenLake ({MAP_NAME}, "
          f"slippery={IS_SLIPPERY}) for {EPISODES} episodes...\n")

    Q, rewards = train()

    print_q_table(Q)

    success_rate = evaluate(Q)
    print(f"\nEvaluation: agent reaches the goal {success_rate:.1%} of the time "
          f"over 100 greedy (no-exploration) episodes.")

    if success_rate > 0.7:
        print("✓ Looks correct — the agent has clearly learned a good policy.")
    else:
        print("⚠ Success rate is low. Possible causes: too few episodes, "
              "epsilon not decaying enough, or a bug in the update rule. "
              "Discuss with your teammate before moving to Phase 2.")

Training Q-learning on FrozenLake (4x4, slippery=False) for 5000 episodes...

Episode   500 | epsilon=0.901 | success rate (last 500 eps) = 2.40%
Episode  1000 | epsilon=0.802 | success rate (last 500 eps) = 5.80%
Episode  1500 | epsilon=0.703 | success rate (last 500 eps) = 13.00%
Episode  2000 | epsilon=0.604 | success rate (last 500 eps) = 20.00%
Episode  2500 | epsilon=0.505 | success rate (last 500 eps) = 31.20%
Episode  3000 | epsilon=0.406 | success rate (last 500 eps) = 43.00%
Episode  3500 | epsilon=0.307 | success rate (last 500 eps) = 60.00%
Episode  4000 | epsilon=0.208 | success rate (last 500 eps) = 69.80%
Episode  4500 | epsilon=0.109 | success rate (last 500 eps) = 78.60%
Episode  5000 | epsilon=0.010 | success rate (last 500 eps) = 93.80%

Final Q-table:
State |    LEFT |    DOWN |   RIGHT |      UP
    0 |   0.941 |   0.951 |   0.951 |   0.941
    1 |   0.941 |   0.000 |   0.961 |   0.951
    2 |   0.951 |   0.970 |   0.951 |   0.961
    3 |   0.961 |   0.000 |   0.95

EXPLORATION STRATEGIES

In [19]:

def linear_decay(start, end, step, total):
    return max(end, start - (start - end) * step / total)


def epsilon_greedy_action(Q, state, eps):
    if np.random.rand() < eps:
        return np.random.randint(Q.shape[1])
    return np.argmax(Q[state])


def softmax_action(Q, state, tau):
    q = Q[state] / max(tau, 1e-10)
    q -= q.max()                         # numerical stability
    probs = np.exp(q) / np.exp(q).sum()
    return np.random.choice(len(probs), p=probs)


def thompson_action(mu, sigma, state):
    samples = np.random.normal(mu[state], np.sqrt(sigma[state]))
    return np.argmax(samples)

In [20]:
# SINGLE RUN
# ══════════════════════════════════════════════════════════════════════════════

def run_episode(env, strategy, Q, mu, sigma, counts, ep, total_eps):
    state, _ = env.reset()
    total_reward = 0.0

    param = linear_decay(
        EPS_START if strategy != "softmax" else TAU_START,
        EPS_END   if strategy != "softmax" else TAU_END,
        ep, total_eps
    )

    while True:
        # Action selection
        if strategy == "epsilon_greedy":
            action = epsilon_greedy_action(Q, state, param)
        elif strategy == "softmax":
            action = softmax_action(Q, state, param)
        else:  # thompson
            action = thompson_action(mu, sigma, state)

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # Q-learning update
        td_target = reward + GAMMA * np.max(Q[next_state]) * (not terminated)
        td_error  = td_target - Q[state, action]
        Q[state, action] += ALPHA * td_error

        # Thompson Sampling posterior update (incremental Gaussian)
        if strategy == "thompson":
            counts[state, action] += 1
            n = counts[state, action]
            mu[state, action]    += (Q[state, action] - mu[state, action]) / n
            sigma[state, action]  = 1.0 / (n + 1)

        total_reward += reward
        state = next_state
        if done:
            break

    return total_reward


def run_training(map_name, n_episodes, is_slippery, strategy, seed):
    np.random.seed(seed)
    env = gym.make("FrozenLake-v1", map_name=map_name, is_slippery=is_slippery)
    n_states  = env.observation_space.n
    n_actions = env.action_space.n

    Q      = np.zeros((n_states, n_actions))
    mu     = np.zeros((n_states, n_actions))
    sigma  = np.ones((n_states, n_actions))
    counts = np.zeros((n_states, n_actions), dtype=int)

    rewards = []
    for ep in range(n_episodes):
        r = run_episode(env, strategy, Q, mu, sigma, counts, ep, n_episodes)
        rewards.append(r)

    env.close()
    return np.array(rewards)


# ══════════════════════════════════════════════════════════════════════════════
# METRICS
# ══════════════════════════════════════════════════════════════════════════════

def convergence_episode(rewards_2d, threshold=CONV_THRESH, window=CONV_WINDOW):
    """Episode index at which rolling mean first stays above threshold."""
    mean_curve = rewards_2d.mean(axis=0)
    for i in range(len(mean_curve) - window):
        if mean_curve[i:i+window].mean() >= threshold:
            return i
    return -1   # did not converge


def stability_variance(rewards_2d):
    n = rewards_2d.shape[1]
    tail = rewards_2d[:, int(n * (1 - STABILITY_WIN)):]
    return tail.mean(axis=1).var()


def bootstrap_ci(data_1d, n_boot=CI_BOOTS, ci=95):
    boots = np.array([np.random.choice(data_1d, len(data_1d), replace=True).mean()
                      for _ in range(n_boot)])
    lo = np.percentile(boots, (100 - ci) / 2)
    hi = np.percentile(boots, 100 - (100 - ci) / 2)
    return lo, hi



In [ ]:

def statistical_analysis(results_dict, label):
    """Kruskal-Wallis + pairwise Mann-Whitney U (Bonferroni) on final rewards."""
    strategies = list(results_dict.keys())
    final_rewards = {s: results_dict[s][:, -int(results_dict[s].shape[1]*0.2):].mean(axis=1)
                     for s in strategies}

    H, p_kw = stats.kruskal(*[final_rewards[s] for s in strategies])
    print(f"\n  [{label}] Kruskal-Wallis H={H:.3f}, p={p_kw:.4f}")

    pairs = [(strategies[i], strategies[j])
             for i in range(len(strategies)) for j in range(i+1, len(strategies))]
    bonferroni = ALPHA_STAT / len(pairs)

    rows = []
    for a, b in pairs:
        U, p_mw = stats.mannwhitneyu(final_rewards[a], final_rewards[b], alternative='two-sided')
        r_effect = 1 - 2*U / (len(final_rewards[a]) * len(final_rewards[b]))
        sig = "✓" if p_mw < bonferroni else "✗"
        print(f"    {LABELS[a]} vs {LABELS[b]}: U={U:.0f}, p={p_mw:.4f} {sig} (r={r_effect:.3f})")
        rows.append({
            "comparison": f"{LABELS[a]} vs {LABELS[b]}",
            "U": round(U, 2), "p": round(p_mw, 4),
            "effect_r": round(r_effect, 3),
            "significant": bool(p_mw < bonferroni) # Convert boolean to string
        })
    return {"H": round(H, 3), "p_kruskal": round(p_kw, 4), "pairwise": rows}


# ══════════════════════════════════════════════════════════════════════════════
# PLOTTING
# ══════════════════════════════════════════════════════════════════════════════

def smooth(x, w=50):
    return np.convolve(x, np.ones(w)/w, mode='valid')


def plot_learning_curves(results_dict, title, filename):
    if not PLOT:
        return
    fig, ax = plt.subplots(figsize=(10, 5))
    for s, color in COLORS.items():
        r = results_dict[s]
        mean = smooth(r.mean(axis=0))
        # CI via bootstrap on smoothed curve
        lo = smooth(np.percentile(r, 2.5, axis=0))
        hi = smooth(np.percentile(r, 97.5, axis=0))
        x = np.arange(len(mean))
        ax.plot(x, mean, color=color, label=LABELS[s], linewidth=2)
        ax.fill_between(x, lo, hi, color=color, alpha=0.15)

    ax.set_xlabel("Episode", fontsize=12)
    ax.set_ylabel("Mean Cumulative Reward (smoothed, w=50)", fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-0.05, 1.05)
    plt.tight_layout()
    plt.savefig(f"results/{filename}.pdf", dpi=200, bbox_inches='tight')
    plt.savefig(f"results/{filename}.png", dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved results/{filename}.png/.pdf")


def plot_convergence_scatter(all_metrics, filename="convergence_scatter"):
    if not PLOT:
        return
    fig, ax = plt.subplots(figsize=(8, 6))
    markers = {'4x4_det': 'o', '4x4_sto': 's', '8x8_det': '^', '8x8_sto': 'D'}
    for cond, strategies in all_metrics.items():
        for s, m in strategies.items():
            conv = m['convergence_ep']
            stab = m['stability_var']
            if conv == -1:
                conv = m['n_episodes']
            ax.scatter(conv, stab, color=COLORS[s], marker=markers.get(cond, 'o'),
                       s=120, alpha=0.8,
                       label=f"{LABELS[s]} ({cond})" if s == 'epsilon_greedy' else "")
    ax.set_xlabel("Convergence Episode", fontsize=12)
    ax.set_ylabel("Policy Stability (variance of final reward)", fontsize=12)
    ax.set_title("Convergence Speed vs. Policy Stability", fontsize=14, fontweight='bold')
    patches = [mpatches.Patch(color=c, label=LABELS[s]) for s, c in COLORS.items()]
    ax.legend(handles=patches, fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"results/{filename}.pdf", dpi=200, bbox_inches='tight')
    plt.savefig(f"results/{filename}.png", dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved results/{filename}.png/.pdf")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN EXPERIMENT LOOP
# ══════════════════════════════════════════════════════════════════════════════

def main():
    strategies   = ["epsilon_greedy", "softmax", "thompson"]
    grid_sizes   = ["4x4", "8x8"]
    stochastics  = [False, True]
    stoch_labels = {False: "det", True: "sto"}

    all_results = {}   # condition → strategy → (N_RUNS × episodes) array
    all_metrics = {}   # condition → strategy → metric dict
    all_stats   = {}   # condition → stat test results

    for grid, slippery in product(grid_sizes, stochastics):
        cond  = f"{grid}_{stoch_labels[slippery]}"
        n_eps = CONFIG[grid]["episodes"]
        label = f"{grid} | {'stochastic' if slippery else 'deterministic'}"
        print(f"\n{'='*60}")
        print(f"  Condition: {label}")
        print(f"{'='*60}")

        all_results[cond] = {}
        all_metrics[cond] = {}

        for s in strategies:
            print(f"  Running {LABELS[s]} × {N_RUNS} seeds ...", end=" ", flush=True)
            runs = np.array([
                run_training(grid, n_eps, slippery, s, seed=i)
                for i in range(N_RUNS)
            ])
            all_results[cond][s] = runs

            # Compute metrics
            final_rewards = runs[:, -int(n_eps * STABILITY_WIN):].mean(axis=1)
            mean_fr       = final_rewards.mean()
            ci_lo, ci_hi  = bootstrap_ci(final_rewards)
            conv_ep       = convergence_episode(runs)
            stab_var      = stability_variance(runs)

            all_metrics[cond][s] = {
                "mean_final_reward": round(mean_fr, 4),
                "ci_95": (round(ci_lo, 4), round(ci_hi, 4)),
                "convergence_ep": conv_ep,
                "stability_var": round(stab_var, 6),
                "n_episodes": n_eps,
            }
            print(f"mean={mean_fr:.3f} [{ci_lo:.3f}, {ci_hi:.3f}]  conv={conv_ep}  stab={stab_var:.5f}")

        # Plot learning curves for this condition
        plot_learning_curves(
            all_results[cond],
            title=f"Learning Curves — FrozenLake {label}",
            filename=f"curves_{cond}"
        )

        # Statistical tests
        all_stats[cond] = statistical_analysis(all_results[cond], label)

    # Convergence scatter across all conditions
    plot_convergence_scatter(all_metrics)

    # ── Save summary tables ──────────────────────────────────────────────────
    print("\n\n" + "="*60)
    print("  SUMMARY TABLE")
    print("="*60)
    print(f"{'Condition':<20} {'Strategy':<22} {'Mean Reward':<15} {'95% CI':<22} {'Conv. Ep':<12} {'Stability'}")
    print("-"*100)
    for cond, strats in all_metrics.items():
        for s, m in strats.items():
            ci = f"[{m['ci_95'][0]:.3f}, {m['ci_95'][1]:.3f}]"
            conv = str(m['convergence_ep']) if m['convergence_ep'] != -1 else "No conv."
            print(f"{cond:<20} {LABELS[s]:<22} {m['mean_final_reward']:<15.4f} {ci:<22} {conv:<12} {m['stability_var']:.6f}")

    # Save JSON for reference
    with open("results/all_metrics.json", "w") as f:
        json.dump(all_metrics, f, indent=2)
    with open("results/statistical_tests.json", "w") as f:
        json.dump(all_stats, f, indent=2)

    print("\n✓ All results saved in results/")
    print("✓ Plots: results/curves_*.png  and  results/convergence_scatter.png")
    print("✓ Data:  results/all_metrics.json  and  results/statistical_tests.json")
    print("\nNext steps:")
    print("  1. Copy plots into the Word document at the [INSERT] placeholders")
    print("  2. Fill in the [FILL IN] sections with your actual observed numbers")
    print("  3. Run on a GPU/faster machine if 8x8 stochastic is slow")


if __name__ == "__main__":
    main()


  Condition: 4x4 | deterministic
  Running Epsilon-Greedy × 30 seeds ... mean=0.884 [0.880, 0.887]  conv=3666  stab=0.00008
  Running Softmax × 30 seeds ... mean=0.881 [0.878, 0.885]  conv=3880  stab=0.00010
  Running Thompson Sampling × 30 seeds ... mean=0.767 [0.600, 0.900]  conv=718  stab=0.17883
  Saved results/curves_4x4_det.png/.pdf

  [4x4 | deterministic] Kruskal-Wallis H=17.522, p=0.0002
    Epsilon-Greedy vs Softmax: U=518, p=0.3142 ✗ (r=-0.152)
    Epsilon-Greedy vs Thompson Sampling: U=210, p=0.0003 ✓ (r=0.533)
    Softmax vs Thompson Sampling: U=210, p=0.0003 ✓ (r=0.533)

  Condition: 4x4 | stochastic
  Running Epsilon-Greedy × 30 seeds ... mean=0.393 [0.387, 0.398]  conv=-1  stab=0.00028
  Running Softmax × 30 seeds ... mean=0.172 [0.166, 0.178]  conv=-1  stab=0.00027
  Running Thompson Sampling × 30 seeds ... mean=0.689 [0.658, 0.712]  conv=3476  stab=0.00561
  Saved results/curves_4x4_sto.png/.pdf

  [4x4 | stochastic] Kruskal-Wallis H=79.143, p=0.0000
    Epsilon-Gree